# Desktop Helper — SmolLM2 era fine-tuning

Fine-tunes `DesktopHelperLM` (modern Llama-style architecture, SmolLM2-1.7B-Instruct weights transplanted) on Dolly + tool-call data.

**Runtime:** GPU required (Runtime → Change runtime type). Free-tier T4/L4 (~15GB) works at batch 1 + grad-accum; an A100 (Colab Pro) is comfortable.

**Drive space:** each epoch checkpoint is ~3.4GB (bf16, no optimizer state). Clear old OPT-era checkpoints from `desktop_helper_checkpoints/` if space is tight.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo (smollm2-1.7b branch)

In [ ]:
!git clone https://github.com/JaysonSalemmo/Desktop_Helper.git
%cd Desktop_Helper
!git checkout smollm2-1.7b

## 3. Install dependencies

Colab ships torch + CUDA. bitsandbytes provides the 8-bit optimizer that makes 1.7B fit on a T4.

In [ ]:
!pip install -q datasets tokenizers tensorboard transformers 'bitsandbytes>=0.44'

## 4. Transplant SmolLM2 → DesktopHelperLM (in-notebook)

Downloads ~3.4GB on datacenter bandwidth and writes the fp32 transplant to local disk (not Drive — it's rebuilt each session in ~2 min).

In [ ]:
!python -m model.load_base --output /content/smol_transplant.pt

## 5. Safety gate: logit-equivalence test

The OPT-era lesson: a transplant bug once cost a full training run to discover. This compares our architecture against HuggingFace's reference — **do not train if this fails.**

In [ ]:
!ln -sf /content/smol_transplant.pt model/checkpoints/smol_transplant.pt
!python -m pytest tests/test_transplant.py -q

## 6. Regenerate the tool-call data

Deterministic (same seed as local). Includes the no-tool chat category (routing contrast).

In [ ]:
!python -m model.data.tool_calls --count 4000 --seed 42 --output data/tool_calls.jsonl

## 7. Launch TensorBoard (run before training)

Watch `loss/train` and — new this era — `loss/held_out`: the catastrophic-forgetting alarm. If held-out loss climbs while train loss falls, stop and reduce epochs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/runs

## 8. Fine-tune

Light-touch adaptation: 3 epochs, single LR 2e-5, pure bf16, 8-bit AdamW, gradient checkpointing. Checkpoints go straight to Drive under `smol_run1/` (each run gets its own folder — no more overwrite trap).

If OOM on a T4: batch-size is already 1; reduce `--grad-accum` won't help memory — close other notebooks or use an A100.

In [ ]:
!python -m model.train \
  --checkpoint /content/smol_transplant.pt \
  --tokenizer model/hf_tokenizer \
  --tool-calls data/tool_calls.jsonl \
  --output /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/ \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 32

## 9. Faithfulness eval

Scores RESULT-copying on held-out entities. OPT-era final score was 88% (run 5); the transplant's pretrained embeddings should start far higher.

In [ ]:
!python -m model.eval_faithfulness --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/epoch_03.pt --tokenizer model/hf_tokenizer

## 10. Chat sanity — the Ojomala Adar memorial cell

The entire point of this era: "Hello" must produce sense.

In [ ]:
for prompt in ["Hello!", "How are you today?", "What can you do?", "Tell me a fun fact about space."]:
    !python -m model.generate --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/smol_run1/epoch_03.pt --tokenizer model/hf_tokenizer --prompt "{prompt}" --max-new-tokens 60